# Method 2: Transformer-based Text Classification

## 1. Load Libraries

In [ ]:
import math
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertConfig
from sklearn.metrics import precision_score, recall_score, f1_score

## 2. Construct Data for Trasnformer-based Model

### 2.1 Load the Processed Data

In [ ]:
PATH_PREFIX = '../'

data_tran = pd.read_json(PATH_PREFIX + 'data/data_processed/data_tran.json', orient='index')
data_test = pd.read_json(PATH_PREFIX + 'data/data_processed/data_test.json', orient='index')

In [ ]:
data_tran = data_tran.head(200)
data_test = data_test.head(50)

In [ ]:
n_tran = data_tran.shape[0]
n_test = data_test.shape[0]

In [ ]:
data_tran.head(3)

In [ ]:
data_test.head(3)

### 2.2 Convert Year and Venue into Single-Element List & Add to Vocabulary

In [ ]:
def covert_int_to_list(num, add_value):
    return [num + add_value]

def convert_list_to_list(num_list, add_value):
    return [num + add_value for num in num_list]

In [ ]:
data_tran['year'] = data_tran['year'].apply(lambda x: covert_int_to_list(x, 5000))
data_test['year'] = data_test['year'].apply(lambda x: covert_int_to_list(x, 5000))

In [ ]:
data_tran['venue'] = data_tran['venue'].apply(lambda x: covert_int_to_list(x, 5020))
data_test['venue'] = data_test['venue'].apply(lambda x: covert_int_to_list(x, 5020))

### 2.3 Convert the List of Co-authors & Add Co-authors to the Vocabulary

In [ ]:
data_tran['authors_coop'] = data_tran['authors_coop'].apply(lambda x: convert_list_to_list(x, 5486))
data_test['authors_coop'] = data_test['authors_coop'].apply(lambda x: convert_list_to_list(x, 5486))

### 2.4 Create a Combined List of All Features

In [ ]:
data_tran['list_all'] = data_tran['year'] + data_tran['venue'] + data_tran['authors_coop'] + data_tran['list_combined']
data_test['list_all'] = data_test['year'] + data_test['venue'] + data_test['authors_coop'] + data_test['list_combined']

### 2.5 Show the Processed Data

In [ ]:
data_vald = data_tran.sample(frac=0.2, random_state=42)
data_tran = data_tran.drop(data_vald.index)

In [ ]:
data_tran.head(3)

In [ ]:
data_vald.head(3)

In [ ]:
data_test.head(3)

## 3. BERT-based Text Classification

### 3.1 Dataset & DataLoader

In [ ]:
y_tran = np.load(PATH_PREFIX + "/data/data_processed/y_tran.npy")
y_tran = y_tran[:n_tran]
y_vald = y_tran[data_vald.index]
y_tran = y_tran[data_tran.index]

In [ ]:
class MethodDataset(Dataset):
    def __init__(self, data, labels=None, max_length=512):
        self.data = data
        self.labels = labels
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data.iloc[idx]['list_all']
        # Truncate if longer than max_length
        if len(x) > self.max_length:
            x = x[:self.max_length]
        # Pad with 0 if shorter
        x += [0] * (self.max_length - len(x))
        x = torch.tensor(x, dtype=torch.long)
        attention_mask = (x != 0).long()  # 1 for real tokens, 0 for padding
        if self.labels is not None:
            y = torch.tensor(self.labels[idx], dtype=torch.long)
            return x, attention_mask, y
        else:
            return x, attention_mask

In [ ]:
dataset_tran = MethodDataset(data_tran, y_tran)
dataset_vald = MethodDataset(data_vald, y_vald)
dataset_test = MethodDataset(data_test)

In [ ]:
dataloader_tran = DataLoader(dataset_tran, batch_size=4, shuffle=True)
dataloader_vald = DataLoader(dataset_vald, batch_size=4, shuffle=False)
dataloader_test = DataLoader(dataset_test, batch_size=4, shuffle=False)

### 3.2 Model Design

In [ ]:
class BertClassifier(nn.Module):
    def __init__(self, num_classes=101):
        super().__init__()
        config = BertConfig()
        self.bert = BertModel(config)  # 随机初始化，无预训练权重
        self.classifier = nn.Linear(config.hidden_size, num_classes)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # 使用[CLS] token的输出
        logits = self.classifier(pooled_output)
        probs = self.sigmoid(logits)
        return probs

### 3.3 Model Training & Evaluation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BertClassifier(num_classes=101).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=0.00001)

In [ ]:
def calculate_metrics(pred_label, true_label):
    pred_label = pred_label.int()
    true_label = true_label.int()
    pc = precision_score(true_label.cpu(), pred_label.cpu(), average='macro', zero_division=0)
    rc = recall_score(true_label.cpu(), pred_label.cpu(), average='macro', zero_division=0)
    f1 = f1_score(true_label.cpu(), pred_label.cpu(), average='macro', zero_division=0)
    return pc, rc, f1

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, train_loss):
        if self.best_loss is None or train_loss < self.best_loss - self.delta:
            self.best_loss = train_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStopping(patience=10, delta=0.001)

In [ ]:
epochs = 10

for epoch in range(epochs):
    
    model.train()
    train_loss = 0.0
    
    for batch in dataloader_tran:
        input_ids, attention_mask, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device).float()
        
        optimizer.zero_grad()
        
        y_tran_pred_prob = model(input_ids, attention_mask)
        loss = criterion(y_tran_pred_prob, labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(dataloader_tran)
    
    if (epoch + 1) % 10 == 0:
        model.eval()
        
        # Train metrics
        all_tran_preds = []
        all_tran_labels = []
        with torch.no_grad():
            for batch in dataloader_tran:
                input_ids, attention_mask, labels = batch
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device).float()
                
                y_tran_pred_prob = model(input_ids, attention_mask)
                y_tran_pred_labl = (y_tran_pred_prob > 0.5).int()
                
                all_tran_preds.append(y_tran_pred_labl.cpu())
                all_tran_labels.append(labels.cpu())
        
        all_tran_preds = torch.cat(all_tran_preds, dim=0)
        all_tran_labels = torch.cat(all_tran_labels, dim=0)
        tran_pc, tran_rc, tran_f1 = calculate_metrics(all_tran_preds, all_tran_labels)
        
        # Val metrics
        all_vald_preds = []
        all_vald_labels = []
        with torch.no_grad():
            for batch in dataloader_vald:
                input_ids, attention_mask, labels = batch
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device).float()
                
                y_vald_pred_prob = model(input_ids, attention_mask)
                y_vald_pred_labl = (y_vald_pred_prob > 0.5).int()
                
                all_vald_preds.append(y_vald_pred_labl.cpu())
                all_vald_labels.append(labels.cpu())
        
        all_vald_preds = torch.cat(all_vald_preds, dim=0)
        all_vald_labels = torch.cat(all_vald_labels, dim=0)
        vald_pc, vald_rc, vald_f1 = calculate_metrics(all_vald_preds, all_vald_labels)
        
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss:.4f}")
        print(f"Train - Precision: {tran_pc:.4f}, Recall: {tran_rc:.4f}, F1 Score: {tran_f1:.4f}")
        print(f"Val   - Precision: {vald_pc:.4f}, Recall: {vald_rc:.4f}, F1 Score: {vald_f1:.4f}")
        print()
        
        early_stopping(train_loss)
        if early_stopping.early_stop:
            print("Early Stop !")
            break

### 3.4 Model Inference on the Test Set

In [ ]:
model.eval()
all_test_preds = []
all_test_input_ids = []
with torch.no_grad():
    for batch in dataloader_test:
        input_ids, attention_mask = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        
        y_test_pred_prob = model(input_ids, attention_mask)
        y_test_pred_labl = (y_test_pred_prob > 0.5).int()
        
        all_test_preds.append(y_test_pred_labl.cpu())
        all_test_input_ids.append(input_ids.cpu())

y_test_pred_labl = torch.cat(all_test_preds, dim=0)
x_test = torch.cat(all_test_input_ids, dim=0)

In [ ]:
def generate_output_csv(x_test, y_test_pred_labl):
    
    result = []
    
    for i, row in enumerate(y_test_pred_labl):
        
        if (x_test[i, :100] == 0).all() or (x_test[i, 100:200] == 0).all() or (x_test[i, 200:300] == 0).all() or (x_test[i, 300:400] == 0).all() or (x_test[i, 400:500] == 0).all():
            result.append("-1")
        elif row.sum() == 0 or row[100] == 1:
            result.append("-1")
        else:
            indices = [str(idx) for idx, val in enumerate(row) if val == 1]
            result.append(" ".join(indices))
    
    result_df = pd.DataFrame({"ID": range(len(result)), "Predict": result})
    
    return result_df

generate_output_csv(x_test, y_test_pred_labl).to_csv("../data/data_result/result_method2.csv", index=False)